# RAG-IDEArq — Evaluación con LangGraph + RAGAS

Evaluación del sistema RAG usando **LangGraph** con nodos:
- `retrieve` → `rerank` (opcional) → `generate` → `evaluate` (RAGAS)

**Dataset v3**: 30 preguntas (15 simples + 15 complejas, multilingüe)

## Uso con LangGraph Studio
```bash
cd /home/raglinux/RAG
source ../env_rag/bin/activate
langgraph dev --port 8123
```
→ Abre `http://localhost:8123` para probar el grafo interactivamente

In [ ]:
# Cell 1: Setup
import os
import sys
import time
import itertools
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any

# Add project root to path
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

from src.config import (
    EMBEDDINGS, LLMS, LLM_TEMPERATURES, RESULTS_DIR,
    RERANKING_CONFIG,
)
from data.eval_questions import (
    get_eval_items, get_metadata, get_questions, get_ground_truths, get_sources,
)

import pandas as pd

print(f"Project root: {PROJECT_ROOT}")
print(f"Embeddings: {list(EMBEDDINGS.keys())}")
print(f"LLMs: {list(LLMS.keys())}")
print(f"Temperatures: {LLM_TEMPERATURES}")
print(f"Reranking enabled: {RERANKING_CONFIG['enabled']}")
print(f"Rerank model: {RERANKING_CONFIG['model']}")

In [ ]:
# Cell 2: Import LangGraph graph
from src.graph_eval import graph

# Verify graph structure
print("Graph nodes:", list(graph.nodes.keys()))
print("Graph edges:", list(graph.edges))

In [ ]:
# Cell 3: Load dataset v3
eval_items = get_eval_items()
print(f"Dataset: RAG-IDEArq-eval-v3")
print(f"Items: {len(eval_items)}")
print(f"\nMetadata: {get_metadata()}")

# Show first question as example
print(f"\nExample question:")
print(f"  Q: {eval_items[0]['input']['question'][:80]}...")
print(f"  Tipo: {eval_items[0]['input']['tipo']}")
print(f"  Idioma: {eval_items[0]['input']['idioma']}")

In [ ]:
# Cell 4: Run evaluation grid
all_results = []

# Experiment grid
combos = list(itertools.product(
    EMBEDDINGS.keys(),
    LLMS.keys(),
    ["zero_shot", "one_shot", "few_shot"],
    LLM_TEMPERATURES,
))

print(f"\nTotal combos: {len(combos)}")
print(f"Questions per combo: {len(eval_items)}")
print(f"Total evaluations: {len(combos) * len(eval_items)}\n")

for emb_key, llm_name, prompt_key, temperature in combos:
    combo_label = f"{emb_key}|{llm_name}|{prompt_key}|t{temperature}"
    print(f"\n{'='*60}")
    print(f"Combo: {combo_label}")
    print(f"{'='*60}")

    for idx, item in enumerate(eval_items):
        question = item["input"]["question"]
        ground_truth = item["expected_output"]["ground_truth"]
        source = item["expected_output"]["source"]
        tipo = item["input"]["tipo"]
        idioma = item["input"]["idioma"]
        n_articulos = item["input"]["n_articulos"]

        print(f"  Q{idx+1}/{len(eval_items)} [{tipo}/{idioma}]: {question[:60]}...")

        t0 = time.time()
        try:
            # Invoke the LangGraph
            result = graph.invoke({
                "question": question,
                "embedding_key": emb_key,
                "llm_name": llm_name,
                "prompt_key": prompt_key,
                "temperature": temperature,
                "use_rerank": RERANKING_CONFIG["enabled"],
                "ground_truth": ground_truth,
            })
            latency = time.time() - t0

            # Extract RAGAS scores from graph output
            ragas_scores = result.get("ragas_scores", {})

            # Store result
            all_results.append({
                "combo": combo_label,
                "embedding": emb_key,
                "llm": llm_name,
                "prompt": prompt_key,
                "temperature": temperature,
                "use_rerank": RERANKING_CONFIG["enabled"],
                "question": question,
                "answer": result.get("answer", "")[:500],
                "ground_truth": ground_truth[:500],
                "source": source,
                "tipo": tipo,
                "idioma": idioma,
                "n_articulos": n_articulos,
                "n_docs_retrieved": len(result.get("retrieved_docs", [])),
                "latency_s": latency,
                "faithfulness": ragas_scores.get("faithfulness", 0),
                "context_precision": ragas_scores.get("context_precision", 0),
                "context_recall": ragas_scores.get("context_recall", 0),
                "answer_correctness": ragas_scores.get("answer_correctness", 0),
            })

            print(f"    OK ({latency:.1f}s, RAGAS: faithfulness={ragas_scores.get('faithfulness', 0):.3f})")

        except Exception as e:
            print(f"    ERROR: {e}")
            all_results.append({
                "combo": combo_label,
                "embedding": emb_key,
                "llm": llm_name,
                "prompt": prompt_key,
                "temperature": temperature,
                "use_rerank": RERANKING_CONFIG["enabled"],
                "question": question,
                "answer": f"Error: {e}",
                "ground_truth": ground_truth[:500],
                "source": source,
                "tipo": tipo,
                "idioma": idioma,
                "n_articulos": n_articulos,
                "n_docs_retrieved": 0,
                "latency_s": 0,
                "faithfulness": 0,
                "context_precision": 0,
                "context_recall": 0,
                "answer_correctness": 0,
            })

    # Pause between combos
    print(f"\n  Waiting 60s before next combo...")
    time.sleep(60)

In [ ]:
# Cell 5: Reporte segmentado
if not all_results:
    print("No results to report.")
else:
    df = pd.DataFrame(all_results)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = Path(RESULTS_DIR) / f"eval_v3_langgraph_{timestamp}.csv"
    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Results saved to: {output_path}")
    print(f"Total rows: {len(df)}")

    # Tabla 1: por tipo
    print("\n" + "="*60)
    print("Resultados por TIPO (simple vs compleja)")
    print("="*60)
    print(df.groupby("tipo")[["faithfulness", "context_recall", "answer_correctness", "context_precision"]].mean())

    # Tabla 2: por idioma
    print("\n" + "="*60)
    print("Resultados por IDIOMA")
    print("="*60)
    print(df.groupby("idioma")[["faithfulness", "context_recall", "answer_correctness", "context_precision"]].mean())

    # Tabla 3: por modelo + tipo
    print("\n" + "="*60)
    print("Answer Correctness por MODELO × TIPO")
    print("="*60)
    print(df.groupby(["llm", "tipo"])["answer_correctness"].mean().unstack())

    # Tabla 4: por embedding + tipo
    print("\n" + "="*60)
    print("Context Precision por EMBEDDING × TIPO")
    print("="*60)
    print(df.groupby(["embedding", "tipo"])["context_precision"].mean().unstack())

    # Tabla 5: latencia por modelo
    print("\n" + "="*60)
    print("Latencia media por MODELO")
    print("="*60)
    print(df.groupby("llm")["latency_s"].mean().sort_values())

    # Tabla 6: por temperatura
    print("\n" + "="*60)
    print("Resultados por TEMPERATURA")
    print("="*60)
    print(df.groupby("temperature")[["faithfulness", "context_recall", "answer_correctness", "context_precision"]].mean())

    # Tabla 7: por rerank
    print("\n" + "="*60)
    print("RESULTADOS POR RERANKING")
    print("="*60)
    if "use_rerank" in df.columns:
        print(df.groupby("use_rerank")[["faithfulness", "context_recall", "answer_correctness", "context_precision"]].mean())

    # Tabla 8: por rerank × idioma
    print("\n" + "="*60)
    print("RESULTADOS POR RERANKING × IDIOMA")
    print("="*60)
    if "use_rerank" in df.columns:
        print(df.groupby(["use_rerank", "idioma"])["answer_correctness"].mean().unstack())

    print("\nDone! Check LangGraph Studio at http://localhost:8123 for detailed traces.")